# Notebook 21 — Mac mini trains, MacBook evaluates

**No cells have been run.** The MacBook needs the prepared MacBook-Evaluation.zip via AirDrop. Unzip there, run SETUP.command once, then START-EVALUATION.command. Keep both Macs awake on the same home network.

1. On the mini, run the relay cell below. It starts only the evaluation queue, and can be used while notebook 18 continues.
2. On the MacBook, start the evaluation worker. It must reproduce 16 frozen-model cases before it is accepted.
3. After notebook 18 finishes AND the MacBook check passes, run the preparation cell. It installs the staged optimizations and prepares a verified resume copy of the latest model.
4. You run the final Start cell. Mini training uses the same 176 decks, six learning threads, four collectors and eight-hour soft budget; the MacBook handles the same 512 evaluation cases per snapshot.

The mini continues training while evaluation runs remotely. Results are labeled with their original snapshot, not the current moving model. Safety-stop detection is delayed until a remote report arrives and is checked at the next training-batch boundary. A backlog of three snapshots stops further training rather than allowing indefinitely stale monitoring. Existing checkpoints remain durable. A worker/network outage preserves queued work and cached cases. Local engine repairs remain staged separately.

This remote mode is prepared, not connected or cross-machine verified yet. Activation refuses until the MacBook passes the exact source/snapshot check. Notebook 20 is a separate learning-rate experiment; run only one training experiment at a time.


In [ ]:
# Owner-run relay only: this does not start training or interrupt notebook 18.
from pathlib import Path
import subprocess, json, urllib.request
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p/'tools/train_neural.py').exists())
PYTHON = ROOT/'.venv/bin/python'
connection = json.loads((ROOT/'runs/remote_evaluation/connection.json').read_text())
request = urllib.request.Request('http://127.0.0.1:8766/health', headers={'Authorization':'Bearer '+connection['token']})
try:
    with urllib.request.urlopen(request,timeout=2) as response:
        assert json.load(response)['protocol'] == 'frozen-evaluation-v1'
    print('Evaluation relay is already running.')
except (OSError, TimeoutError):
    relay_log = open(ROOT/'runs/remote_evaluation/relay.log','a')
    relay_process = subprocess.Popen([str(PYTHON),str(ROOT/'tools/evaluation_relay.py'),'--start'],cwd=ROOT,stdout=relay_log,stderr=subprocess.STDOUT)
    print('Relay launched. Open SETUP.command once, then START-EVALUATION.command on the MacBook.')
print('No training started. Wait for the MacBook equivalence check before running the next cell.')


In [ ]:
from pathlib import Path
import json, subprocess, sys
from IPython.display import HTML, display
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p/'tools/train_neural.py').exists())
if str(ROOT) not in sys.path: sys.path.insert(0,str(ROOT))
from importlib import reload
import hearthstone_sim.neural.dashboard as dashboard_module
Dashboard = reload(dashboard_module).Dashboard
PYTHON = ROOT/'.venv/bin/python'
PREFLIGHT_OK = False
subprocess.run([str(PYTHON),str(ROOT/'tools/activate_remote_evaluation.py'),'--activate'],cwd=ROOT,check=True)
import os
os.environ['HEARTHSTONE_EVALUATION_QUEUE'] = str(ROOT/'runs/remote_evaluation')
campaign_tool = ROOT/'tools/long_training_campaign.py'
# Preparation, integrity checks and configuration validation only; no games or lessons.
preflight = subprocess.run([str(PYTHON),str(campaign_tool)],cwd=ROOT,text=True,stdout=subprocess.PIPE,stderr=subprocess.STDOUT)
print(preflight.stdout.splitlines()[0] if preflight.returncode == 0 else preflight.stdout)
if preflight.returncode == 0: print('Resume checks passed. Training has not started.')
PREFLIGHT_OK = preflight.returncode == 0
if not PREFLIGHT_OK: raise RuntimeError('Campaign checks failed. Training will not start.')


In [ ]:
# This cell starts only when YOU run it. No training was started while preparing this notebook.
if not globals().get('PREFLIGHT_OK',False): raise RuntimeError('Run the successful checks first.')
import time
screen = None
dashboard = None
last_redraw = 0.0
redraw_events = {'started','iteration_saved','tactical_finished','evaluation_started','evaluation_finished','complete','stopped'}
with subprocess.Popen([str(PYTHON),str(campaign_tool),'--train'],cwd=ROOT,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True) as process:
    try:
        for line in process.stdout:
            try: event=json.loads(line)
            except ValueError:
                print(line.rstrip());continue
            if event.get('event')=='campaign_ready':
                print('Continuing saved model at',event['global_game'],'games. Remaining campaign games:',event['remaining_games'])
                if event['complete']:print('Campaign complete. No more training will launch.');continue
                dashboard=Dashboard(event['remaining_games']);screen=display(HTML(dashboard.html()),display_id=True)
                print('Baseline evaluation needed:',event['baseline'])
            elif dashboard is not None:
                dashboard.consume(event)
                now=time.monotonic()
                # Collection events still update internal counters, but redraw only at batch boundaries.
                # Evaluation/lesson heartbeats keep long non-batch phases visibly alive.
                if event.get('event') in redraw_events or (event.get('event') in ('evaluation_game','tactical_progress') and now-last_redraw >= 15):
                    screen.update(HTML(dashboard.html()));last_redraw=now
                if event.get('event') in ('tactical_finished','iteration_saved','evaluation_finished','complete','stopped'):
                    summary={k:event[k] for k in ('event','iteration','global_game','checkpoint','metrics','reason','report','assessment_before','assessment_after') if k in event}
                    print(json.dumps(summary))
        returncode=process.wait()
    except BaseException:
        process.terminate()
        try:process.wait(timeout=10)
        except subprocess.TimeoutExpired:process.kill();process.wait()
        raise
if returncode:raise RuntimeError('Run stopped. Saved checkpoints remain intact; inspect the message above.')
